# 6장 — 여행 일정표: 학습률 스케줄 📅

일정표 경주(고정 vs 코사인)를 재현하고, 서성임 반경의 법칙과 코사인 공식을 숫자로 확인합니다.

**이 노트북의 목표**
1. 같은 노이즈에서 고정 큰/작은 보폭 vs 코사인 감쇠 — 속도와 정착을 둘 다 얻는 일정표
2. 서성임 반경이 보폭과 함께 줄어드는 것(정착의 물리학)
3. 코사인 감쇠 공식의 이정표 (t=T/2 절반, t=T 0)

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch6')

## 1. 실험 ① — 일정표 경주 (같은 노이즈, 다른 일정)

흔들리는 나침반의 골짜기(c=2, σ=1.5)를 120스텝 — 세 일정표가 완전히 같은 노이즈를 겪습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

C, SIGMA, T = 2.0, 1.5, 120

def train(lr_fn, seed=6):
    rng = np.random.default_rng(seed)          # 같은 시드 = 같은 노이즈 — 공정한 경주
    w, losses = 8.0, []
    for t in range(T):
        g = C*w + SIGMA*rng.standard_normal()  # 흔들리는 나침반 (3장)
        w -= lr_fn(t) * g
        losses.append(0.5*C*w*w)
    return np.array(losses)

scheds = {
    '고정 0.3':  lambda t: 0.3,
    '고정 0.02': lambda t: 0.02,
    '코사인 0.3→0': lambda t: 0.3 * 0.5 * (1 + np.cos(np.pi*t/T)),
}
runs = {name: train(fn) for name, fn in scheds.items()}

plt.figure(figsize=(7.5, 3.6))
for (name, l), c in zip(runs.items(), ['tab:red', 'tab:blue', 'tab:green']):
    plt.semilogy(np.maximum(l, 1e-4), label=f'{name} (마지막 평균 {l[-20:].mean():.3f})', color=c)
plt.xlabel('스텝'); plt.ylabel('손실 (log)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('같은 노이즈, 다른 일정표'); plt.tight_layout(); plt.show()

hi, lo, cos_ = (runs[k] for k in scheds)
print(f'30스텝 시점: 고정0.3 {hi[29]:.3f} / 고정0.02 {lo[29]:.3f} / 코사인 {cos_[29]:.3f}')
print(f'마지막 20스텝 평균: 고정0.3 {hi[-20:].mean():.3f} / 고정0.02 {lo[-20:].mean():.3f} / 코사인 {cos_[-20:].mean():.4f}')

assert cos_[29] < 0.2 and hi[29] < 0.2, '초반: 코사인은 큰 보폭 못지않게 빠르다'
assert lo[29] > 3, '고정 0.02는 초반이 하세월'
assert hi[-20:].mean() > 0.3, '고정 0.3은 바닥에서 서성이느라 정착 실패'
assert cos_[-20:].mean() < hi[-20:].mean() / 10, '코사인은 고정 0.3보다 10배 이상 깊이 정착'
assert cos_[-20:].mean() < lo[-20:].mean(), '느긋한 고정 0.02보다도 깊다 — 둘의 장점만'
print('\n✔ 확인: 속도와 정착 — 고정 보폭에겐 트레이드오프, 일정표에겐 둘 다')

## 2. 실험 ② — 서성임 반경의 법칙

이미 바닥(w=0)에 도착한 상태에서 노이즈 걸음을 계속 — 보폭별 서성임 반경을 잽니다.

In [ ]:
def wander_radius(lr, seed=7, steps=3000):
    rng = np.random.default_rng(seed)
    w, ws = 0.0, []
    for t in range(steps):
        w -= lr*(C*w + SIGMA*rng.standard_normal())
        if t >= 200: ws.append(w)
    return np.std(ws)

print(' 보폭 lr   서성임 반경(실측)   이론 σ√(lr/(2c−lr·c²))')
radii = {}
for lr in [0.02, 0.08, 0.32]:
    r = wander_radius(lr)
    theory = SIGMA*np.sqrt(lr/(2*C - lr*C*C))
    radii[lr] = r
    print(f'  {lr:5.2f}   {r:10.4f}        {theory:10.4f}')

assert radii[0.02] < radii[0.08] < radii[0.32], '보폭이 클수록 서성임 반경이 크다'
assert 1.6 < radii[0.08]/radii[0.02] < 2.6, '보폭 4배 → 반경 약 2배 (√보폭 비례)'
print('\n✔ 확인: 정착 반경 ∝ √보폭 — 후반에 보폭을 줄여야 하는 이유의 정량판')

## 3. 실험 ③ — 코사인 감쇠의 이정표

> cos 를 안 배웠어도 괜찮습니다 — cos 는 **1에서 −1까지 부드럽게 내려오는 곡선**이고,
> π 는 그 **반 바퀴의 끝 지점**이라고만 알면 아래 공식이 읽힙니다.

In [ ]:
def cosine_lr(t, lr_max=0.2, T=1000):
    return lr_max * 0.5 * (1 + np.cos(np.pi*t/T))

print(' t       lr(t)   (lr_max=0.2, T=1000)')
for t in [0, 250, 500, 750, 1000]:
    print(f' {t:5d}   {cosine_lr(t):.4f}')

assert abs(cosine_lr(0) - 0.2) < 1e-12, '시작 = lr_max'
assert abs(cosine_lr(500) - 0.1) < 1e-12, '중간 = 정확히 절반'
assert abs(cosine_lr(1000)) < 1e-12, '끝 = 0 — 마지막 걸음은 잔걸음'
print('\n✔ 확인: 두 이정표(중간 절반, 끝 0)만 외우면 코사인 감쇠는 다 읽힌다')

# 보너스 — 워밍업 + 코사인 (실전 표준 모양)
ts = np.arange(1000)
lr_full = np.minimum(1, (ts+1)/100) * 0.2 * 0.5 * (1 + np.cos(np.pi*ts/1000))
plt.figure(figsize=(6, 2.4))
plt.plot(ts, lr_full)
plt.xlabel('스텝'); plt.ylabel('lr'); plt.grid(alpha=0.3)
plt.title('warmup_steps=100 + cosine — GPT급 훈련 논문의 그 곡선')
plt.tight_layout(); plt.show()

## 4. 관찰 정리 + 🤖 AI 연결

- **감쇠의 이유**: 바닥 근처 걸음은 노이즈×보폭 지배 — 서성임 반경 ∝ √보폭 →
  후반 보폭 축소 = 정착의 조건. 고정 큰 보폭은 빠르지만 정착 실패, 작은 보폭은 하세월 —
  **일정표는 시간축으로 둘 다** 얻는다.
- **워밍업의 이유**: 출발 직후의 취약 구간(Adam의 자 예열 전 + 낯선 초기 지형) —
  처음 몇백 스텝만 잔걸음. 설정: `warmup_steps`.
- **코사인 감쇠** lr(t)=lr_max·½(1+cos(πt/T)): 중간 절반, 끝 0. 설정: `lr_scheduler: cosine`.
  스텝 감쇠는 손실 곡선에 계단 무늬를 남긴다(9장 판독).
- 회수: 6부 3장 SFT 설정의 두 줄 완독 — 속도의 다이얼(2~6장) 완성.

**다음 챕터**: 7장 — 과적합 (핵심). 깊이 내려간 그 도착지, 정말 좋은 곳일까?
훈련 손실과 검증 손실이 갈라지는 순간 — 통째로 외운 학생의 비극.